# 01 — Primary Experiment: Baseline Predictive Multiplicity

## 1. Purpose and scope

This notebook creates the baseline dataset-level summary for the main experiments. It aggregates the primary predictive-multiplicity and spatial-clustering metrics across runs using the global Rashomon approximation with \(K=25\).

The notebook focuses on three outputs:

- dataset-level multiplicity summaries,
- baseline spatial clustering metrics such as Moran's \(I\) and LISA HH counts,
- compact thesis-facing tables and the main dataset-comparison figure.

More detailed analyses are handled in later notebooks:

- permutation-null testing: notebook 02,
- hotspot stability and spatial structure: notebook 03,
- robustness to \(K\), kNN size, calibration, and graph construction: notebooks 04, 05, 07, and 10,
- family and hyperparameter analyses: notebook 06,
- synthetic validation: notebook 08,
- interpretable rules: notebook 09.

## 2. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from thesis_layout import display_dataset_name, RAW_RESULTS, thesis_figure_dir, thesis_table_dir

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
from IPython.display import display

from analysis.experiment_runner import run_dataset_experiment, run_all_experiments
from analysis.knn_defaults import K_NN_BY_DATASET
from analysis.run_analysis import load_P_test, select_rashomon_global, run_multiplicity, run_spatial
from scripts.export_thesis_assets import SUPPORTED_DATASETS
from analysis.preprocessing import get_transformed_test_features

RESULTS_DIR = RAW_RESULTS
K = 25
# k for audit shape checks (COMPAS representative run); see 05_sensitivity_kNN + analysis.knn_defaults
K_NN = K_NN_BY_DATASET["compas"]
SEED = 42
DATASET = None  # set to e.g. "compas" or "adult" to run one dataset only
FORCE_RECOMPUTE = False
CACHE_VERSION = "v1"
CACHE_DIR = ROOT / "thesis_outputs" / "cache" / "notebooks"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from analysis.cache import load_or_compute_df


In [2]:
# Load results: run experiment for each dataset
def compute_primary_aggregate():
    if DATASET:
        dataset_dir = RESULTS_DIR / DATASET
        return run_dataset_experiment(dataset_dir, dataset_name=DATASET, verbose=True)
    return run_all_experiments(RESULTS_DIR, datasets=SUPPORTED_DATASETS, verbose=True)


_cache_tag = DATASET if DATASET else "all"
agg_df = load_or_compute_df(
    CACHE_DIR / f"nb01_primary_aggregate_{_cache_tag}_{CACHE_VERSION}.parquet",
    compute_primary_aggregate,
    force=FORCE_RECOMPUTE,
)
agg_df


,dataset,n_runs,mean_variance_mean,mean_variance_std,ambiguity_mean,ambiguity_std,disagreement_rate_mean,disagreement_rate_std,discrepancy_mean,discrepancy_std,...,conflict_n_hh_std,conflict_hh_rate_mean,conflict_hh_rate_std,hh_jaccard_var_conflict_mean,hh_jaccard_var_conflict_std,frac_significant_moran,frac_significant_hh,null_mean,null_std,null_n_hh_mean
0,compas,10,0.001308,0.000306,0.124914,0.012913,0.255231,0.060101,0.065104,0.007366,...,23.404653,0.084546,0.016219,0.078091,0.056827,1.0,1.0,-0.000463,0.005929,0.0
1,german,10,0.005023,0.002129,0.244889,0.067290,0.453713,0.155637,0.137705,0.032563,...,2.406011,0.006500,0.012030,0.118431,0.312343,0.9,0.8,-0.005358,0.014194,0.0
2,adult,10,0.001383,0.000133,0.096142,0.006347,0.142339,0.011467,0.052617,0.006620,...,59.880344,0.064193,0.006130,0.184215,0.051957,1.0,1.0,-0.000080,0.001528,0.0


### Audit: shape checks and extended baseline summary

This cell verifies that the expected aggregate result tables and representative per-run objects can be loaded. It also displays an extended audit table with the available baseline multiplicity and spatial metrics.

The extended table includes both the main variance-based quantities and supplementary diagnostics such as ambiguity, disagreement rate, discrepancy, conflict-based metrics, and the overlap between variance-based and conflict-based HH masks. These diagnostics are useful for checking the stored results, but the main thesis analysis uses predictive variance and its spatial structure as the primary baseline.

In [3]:
# Audit-friendly shape checks
def load_per_run_summary(results_dir: Path, dataset: str) -> pd.DataFrame:
    path = results_dir / dataset / "summary_per_run.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

# Shape checks
print("=== Audit: shape checks ===")
print(f"agg_df shape: {agg_df.shape}")
for ds in agg_df["dataset"].unique():
    pr = load_per_run_summary(RESULTS_DIR, ds)
    if not pr.empty:
        print(f"  {ds} per_run: {pr.shape}")
# Core object shapes (one representative run: compas seed=0)
_run_dir = RESULTS_DIR / "compas" / "seed=0"
if _run_dir.exists():
    _P_test = load_P_test(_run_dir)
    _idx = select_rashomon_global(_run_dir, K=K)
    _P_sel = _P_test[_idx]
    _mult = run_multiplicity(_run_dir, K=K)
    _X = get_transformed_test_features(_run_dir, "compas")
    _sp = run_spatial(_run_dir, _X, K=K, k=K_NN, seed=SEED)
    print(f"  P_test: {_P_test.shape}, P_sel: {_P_sel.shape}")
    print(f"  var_p: {_mult['pointwise_variance'].shape}, conflict: {_mult['pointwise_conflict'].shape}")
    print(f"  HH_mask: {_sp['HH_mask'].shape}, LL_mask: {_sp['LL_mask'].shape}")

# Compact summary table (all baseline metrics for audit)
summary_cols = [
    "dataset", "n_runs",
    "mean_variance_mean", "mean_variance_std",
    "ambiguity_mean", "disagreement_rate_mean", "discrepancy_mean",
    "mean_conflict_mean",
    "moran_i_mean", "moran_i_std",
    "n_hh_mean",
    "conflict_moran_i_mean", "conflict_n_hh_mean",
    "hh_jaccard_var_conflict_mean", "hh_jaccard_var_conflict_std",
    "frac_significant_moran",
]
summary_cols = [c for c in summary_cols if c in agg_df.columns]
display(agg_df[summary_cols])

=== Audit: shape checks ===
agg_df shape: (3, 36)
  compas per_run: (10, 25)
  german per_run: (10, 25)
  adult per_run: (10, 25)
  P_test: (250, 1443), P_sel: (25, 1443)
  var_p: (1443,), conflict: (1443,)
  HH_mask: (1443,), LL_mask: (1443,)


,dataset,n_runs,mean_variance_mean,mean_variance_std,ambiguity_mean,disagreement_rate_mean,discrepancy_mean,mean_conflict_mean,moran_i_mean,moran_i_std,n_hh_mean,conflict_moran_i_mean,conflict_n_hh_mean,hh_jaccard_var_conflict_mean,hh_jaccard_var_conflict_std,frac_significant_moran
0,compas,10,0.001308,0.000306,0.124914,0.255231,0.065104,0.035440,0.210018,0.081427,128.4,0.166665,122.0,0.078091,0.056827,1.0
1,german,10,0.005023,0.002129,0.244889,0.453713,0.137705,0.059280,0.087606,0.038775,5.4,0.048692,1.3,0.118431,0.312343,0.9
2,adult,10,0.001383,0.000133,0.096142,0.142339,0.052617,0.017211,0.074933,0.021909,631.0,0.117646,627.1,0.184215,0.051957,1.0


## 3. Formatted core baseline table

The table below reports a compact formatted summary of the main baseline metrics used in the primary comparison.

Metric definitions:

- **mean_variance**: average observation-wise predictive variance across the selected Rashomon models.
- **mean_conflict**: average decision-level conflict ratio after thresholding predictions at 0.5.
- **moran_i**: global Moran's \(I\) of the predictive-variance field on the feature-space kNN graph.

The table is intentionally compact. Additional audit metrics such as ambiguity, disagreement rate, discrepancy, HH counts, and conflict-based spatial metrics are shown in the extended audit table above or analyzed in later notebooks.

In [4]:
# Baseline metrics table (formatted mean ± std)
def format_mean_std(mean_val: float, std_val: float, decimals: int = 4) -> str:
    return f"{mean_val:.{decimals}f} ± {std_val:.{decimals}f}"

summary_display = agg_df[["dataset", "n_runs"]].copy()
if "mean_variance_mean" in agg_df.columns:
    summary_display["mean_variance"] = agg_df.apply(
        lambda r: format_mean_std(r["mean_variance_mean"], r["mean_variance_std"]), axis=1
    )
if "mean_conflict_mean" in agg_df.columns:
    summary_display["mean_conflict"] = agg_df.apply(
        lambda r: format_mean_std(r["mean_conflict_mean"], r["mean_conflict_std"]), axis=1
    )
if "moran_i_mean" in agg_df.columns:
    summary_display["moran_i"] = agg_df.apply(
        lambda r: format_mean_std(r["moran_i_mean"], r["moran_i_std"]), axis=1
    )
summary_display

,dataset,n_runs,mean_variance,mean_conflict,moran_i
0,compas,10,0.0013 ± 0.0003,0.0354 ± 0.0070,0.2100 ± 0.0814
1,german,10,0.0050 ± 0.0021,0.0593 ± 0.0219,0.0876 ± 0.0388
2,adult,10,0.0014 ± 0.0001,0.0172 ± 0.0011,0.0749 ± 0.0219


## 4. Dataset comparison figure

The figure below compares four baseline quantities across datasets: mean predictive variance, Moran's \(I\), LISA HH count, and mean conflict ratio. Values are aggregated as mean ± standard deviation over runs.

This figure complements the compact table above by adding the HH count, which summarizes the extent of the detected variance-based hotspot region. Detailed per-run spatial diagnostics, hotspot maps, and stability analyses are handled in notebook 03.

**Interpretation.** Positive Moran's \(I\) indicates that observations with similar predictive-variance values tend to be close in the feature-space graph. LISA HH observations are high-variance points whose neighbors also have high variance, and are interpreted as variance-based multiplicity hotspots.

The key distinction is between disagreement magnitude and spatial concentration. Mean predictive variance measures how much near-optimal models disagree on average, while Moran's \(I\) and HH counts describe how that disagreement is organized in feature space.

In [5]:
# Dataset comparison: 2x2 bar charts (Moran's I, HH count, mean variance, mean conflict)
comp_rows = []
for ds in agg_df["dataset"].unique():
    pr = load_per_run_summary(RESULTS_DIR, ds)
    if pr.empty:
        continue
    comp_rows.append({
        "dataset": ds,
        "moran_i_mean": pr["moran_i"].mean(),
        "moran_i_std": pr["moran_i"].std() if len(pr) > 1 else 0,
        "n_hh_mean": pr["n_hh"].mean(),
        "n_hh_std": pr["n_hh"].std() if len(pr) > 1 else 0,
        "mean_var_mean": pr["mean_variance"].mean(),
        "mean_var_std": pr["mean_variance"].std() if len(pr) > 1 else 0,
        "mean_conflict_mean": pr["mean_conflict"].mean(),
        "mean_conflict_std": pr["mean_conflict"].std() if len(pr) > 1 else 0,
    })
comp_df = pd.DataFrame(comp_rows)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
x = np.arange(len(comp_df))
labels = [display_dataset_name(d) for d in comp_df["dataset"].values]

axes[0, 0].bar(x, comp_df["moran_i_mean"], yerr=comp_df["moran_i_std"], capsize=3, color="seagreen")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(labels)
axes[0, 0].set_ylabel("Moran's I")
axes[0, 0].set_title("Moran's I (mean ± std)")

axes[0, 1].bar(x, comp_df["n_hh_mean"], yerr=comp_df["n_hh_std"], capsize=3, color="darkred")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(labels)
axes[0, 1].set_ylabel("HH count")
axes[0, 1].set_title("LISA HH count (mean ± std)")

axes[1, 0].bar(x, comp_df["mean_var_mean"], yerr=comp_df["mean_var_std"], capsize=3, color="steelblue")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(labels)
axes[1, 0].set_ylabel("Mean variance")
axes[1, 0].set_title("Mean variance (mean ± std)")

axes[1, 1].bar(x, comp_df["mean_conflict_mean"], yerr=comp_df["mean_conflict_std"], capsize=3, color="coral")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_ylabel("Mean conflict")
axes[1, 1].set_title("Mean conflict (mean ± std)")
plt.tight_layout()
FIG_DIR_NB01 = thesis_figure_dir("nb01")
fig.savefig(FIG_DIR_NB01 / "dataset_comparison_bars.pdf", bbox_inches="tight")

## 5. Thesis-facing tables

Tables below match the main thesis results section (Tables `dataset_summary` and `aggregate_multiplicity_metrics`).


In [6]:
# Table: multiplicity and spatial metrics by dataset (thesis Table dataset_summary)
def _fmt_pct(mean_val, std_val, decimals=1):
    return f"{100 * mean_val:.{decimals}f}% ± {100 * std_val:.{decimals}f}%"

thesis_dataset_tbl = pd.DataFrame({
    "dataset": [display_dataset_name(d) for d in agg_df["dataset"]],
    "mean_variance": agg_df.apply(
        lambda r: format_mean_std(r["mean_variance_mean"], r["mean_variance_std"]), axis=1
    ),
    "moran_i": agg_df.apply(
        lambda r: format_mean_std(r["moran_i_mean"], r["moran_i_std"], decimals=3), axis=1
    ),
    "n_hh": agg_df.apply(
        lambda r: f"{r['n_hh_mean']:.1f} ± {r['n_hh_std']:.1f}", axis=1
    ),
    "hh_rate": agg_df.apply(
        lambda r: _fmt_pct(r["hh_rate_mean"], r["hh_rate_std"]), axis=1
    ),
})
print("Thesis table: multiplicity and spatial metrics by dataset")
display(thesis_dataset_tbl)

# Table: additional aggregate multiplicity metrics (thesis Table aggregate_multiplicity_metrics)
thesis_multiplicity_tbl = pd.DataFrame({
    "dataset": [display_dataset_name(d) for d in agg_df["dataset"]],
    "ambiguity": agg_df.apply(
        lambda r: format_mean_std(r["ambiguity_mean"], r["ambiguity_std"]), axis=1
    ),
    "disagreement_rate": agg_df.apply(
        lambda r: format_mean_std(r["disagreement_rate_mean"], r["disagreement_rate_std"]), axis=1
    ),
    "discrepancy": agg_df.apply(
        lambda r: format_mean_std(r["discrepancy_mean"], r["discrepancy_std"]), axis=1
    ),
})
print("Thesis table: additional aggregate multiplicity metrics")
display(thesis_multiplicity_tbl)


Thesis table: multiplicity and spatial metrics by dataset


,dataset,mean_variance,moran_i,n_hh,hh_rate
0,COMPAS,0.0013 ± 0.0003,0.210 ± 0.081,128.4 ± 73.6,8.9% ± 5.1%
1,German Credit,0.0050 ± 0.0021,0.088 ± 0.039,5.4 ± 5.4,2.7% ± 2.7%
2,Adult,0.0014 ± 0.0001,0.075 ± 0.022,631.0 ± 239.7,6.5% ± 2.5%


Thesis table: additional aggregate multiplicity metrics


,dataset,ambiguity,disagreement_rate,discrepancy
0,COMPAS,0.1249 ± 0.0129,0.2552 ± 0.0601,0.0651 ± 0.0074
1,German Credit,0.2449 ± 0.0673,0.4537 ± 0.1556,0.1377 ± 0.0326
2,Adult,0.0961 ± 0.0063,0.1423 ± 0.0115,0.0526 ± 0.0066


## 6. Export summary tables

CSV exports for thesis asset generation (unchanged filenames and columns).


In [7]:
# Optional: save thesis tables
THESIS_TABLES_DIR = thesis_table_dir("nb01")

agg_df.to_csv(THESIS_TABLES_DIR / "dataset_summary.csv", index=False)
print(f"Saved: {THESIS_TABLES_DIR / 'dataset_summary.csv'}")

# Formatted table (mean ± std)
formatted = agg_df[["dataset", "n_runs"]].copy()

for m, s, label in [
    ("mean_variance_mean", "mean_variance_std", "mean_variance"),
    ("moran_i_mean", "moran_i_std", "moran_i"),
    ("hh_jaccard_var_conflict_mean", "hh_jaccard_var_conflict_std", "hh_jaccard_var_conflict"),
]:
    if m in agg_df.columns and s in agg_df.columns:
        formatted[label] = agg_df.apply(lambda r: format_mean_std(r[m], r[s]), axis=1)

formatted.to_csv(THESIS_TABLES_DIR / "dataset_summary_formatted.csv", index=False)
print(f"Saved: {THESIS_TABLES_DIR / 'dataset_summary_formatted.csv'}")

formatted

Saved: C:\Users\dejvi\Documents\pythonProject\Rashomon Sets\rashomon-multiplicity\thesis_outputs\tables\nb01\dataset_summary.csv
Saved: C:\Users\dejvi\Documents\pythonProject\Rashomon Sets\rashomon-multiplicity\thesis_outputs\tables\nb01\dataset_summary_formatted.csv


,dataset,n_runs,mean_variance,moran_i,hh_jaccard_var_conflict
0,compas,10,0.0013 ± 0.0003,0.2100 ± 0.0814,0.0781 ± 0.0568
1,german,10,0.0050 ± 0.0021,0.0876 ± 0.0388,0.1184 ± 0.3123
2,adult,10,0.0014 ± 0.0001,0.0749 ± 0.0219,0.1842 ± 0.0520
